### This script is for creating a data set for the fusion model fine-tuning

In [ ]:
# LIBRARIES
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import load_dataset
from datasets import Dataset
import os
import shutil
import zipfile

# DEFINE WORKING DIRECTORY
# os.chdir("C:/[...]/Rakuten-Ecommerce-Proj")

# DEFINE RANDOM STATE
random_state = 42

In [ ]:
# HELPER FUNCTIONS FOR OPENING ZIP-FILES
zip_file_path = "./data/preprocessed/images.zip"
extract_to = "./data/preprocessed/"

if not os.path.exists(extract_to):
    os.makedirs(extract_to)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

In [ ]:
# CREATE DICTIONARY FOR IMAGE PATHS
def create_path_dict(train_test_path):

    images = {}

    for folder in os.listdir(train_test_path):
        split_path = train_test_path + folder
        for label in  os.listdir(split_path):
            label_path = split_path + "/" + label
            for image in os.listdir(label_path):
                image_path = label_path + "/" + image
                
                images.update({
                    image: image_path
                })

    return images

train_test_path = "./data/preprocessed/images/"
images = create_path_dict(train_test_path)

In [ ]:
# GET IMAGE FUNCTION
def get_image(file_name, images):
    try:
        image = images[file_name]
    
    except:
        image = np.nan

    return image

In [ ]:
# SIMPLE PREPROCESSING

import unicodedata
import re

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    w = unicode_to_ascii(w.lower().strip())

    # creating a space between a word and the punctuation following it
    w = re.sub(r"([?;!,¿])", r" \1 ", w)
    w = re.sub(r'[" "]+', " ", w)

    # replacing everything with space except (a-z, A-Z, ".", "?", "!", ",")
    w = re.sub(r"[^a-zA-Z0-9.°()]+", " ", w)

    return w

In [ ]:
# GET INDICES FOR IMAGES FROM RAW X_TRAIN
image_ids = pd.read_csv("./data/raw/X_train.csv", index_col=0)
image_ids["image_file"] = "image_" + image_ids["imageid"].astype(str) + "_product_" + image_ids["productid"].astype(str) + ".jpg"
image_ids.drop(columns=["designation","description", "productid", "imageid"], inplace=True)
image_ids

In [ ]:
# TRAIN
X_train = pd.read_csv("./modeling/X_train_GGL.csv", index_col=0)
X_train.drop(columns="text", inplace=True)

y_train = pd.read_csv("./data/raw/Y_train.csv", index_col=0)
y_train = y_train.iloc[X_train.index]
image_ids_train = image_ids.iloc[X_train.index]
X_train = X_train.join(image_ids_train)

# VALIDATION
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
df_train = X_train.join(y_train).reset_index(drop=True).rename(columns={"text_GGL": "text", "prdtypecode": "label"})
df_val = X_val.join(y_val).reset_index(drop=True).rename(columns={"text_GGL": "text", "prdtypecode": "label"})

# TEST
X_test = pd.read_csv("./modeling/X_test_GGL.csv", index_col=0)
X_test.drop(columns="text", inplace=True)

y_test = pd.read_csv("./data/raw/Y_train.csv", index_col=0)
y_test = y_test.iloc[X_test.index]
image_ids_test = image_ids.iloc[X_test.index]
X_test = X_test.join(image_ids_test)

df_test = X_test.join(y_test).reset_index(drop=True).rename(columns={"text_GGL": "text", "prdtypecode": "label"})

# GET IMAGE NAMES
df_train["image"] = df_train["image_file"].apply(lambda row: get_image(row,images))
df_val["image"] = df_val["image_file"].apply(lambda row: get_image(row,images))
df_test["image"] = df_test["image_file"].apply(lambda row: get_image(row,images))

# REMOV "image_file" COLUMN
df_train = df_train.drop(columns="image_file")
df_val = df_val.drop(columns="image_file")
df_test = df_test.drop(columns="image_file")

# DROPNA
df_train = df_train.dropna()
df_val = df_val.dropna()
df_test = df_test.dropna()

# REMOVE ROWS WITH ";"
df_train = df_train[(df_train["text"].duplicated() == False) & (df_train["text"] != ";")]
df_val = df_val[(df_val["text"].duplicated() == False) & (df_val["text"] != ";")]
df_test = df_test[(df_test["text"].duplicated() == False) & (df_test["text"] != ";")]

# SELECT RANDOM SAMPLES TO BALANCE MAJORITY CALSS
# df_train[df_train["label"]==2583] = df_train[df_train["label"]==2583].sample(n=3200, random_state=random_state)
# df_train = df_train.dropna()
df_train["label"] = df_train["label"].astype(int)

# CLEAN TEXT
df_train.text = df_train.text.apply(lambda x: preprocess_sentence(x))
df_val.text = df_val.text.apply(lambda x: preprocess_sentence(x))
df_test.text = df_test.text.apply(lambda x: preprocess_sentence(x))

# ENCODING
label_encoder = LabelEncoder()
df_train["label"] = label_encoder.fit_transform(df_train["label"])
df_val["label"] = label_encoder.transform(df_val["label"])
df_test["label"] = label_encoder.transform(df_test["label"])

In [ ]:
# CREATE DATASETS
ds_train = Dataset.from_pandas(df_train, preserve_index=False)
ds_val = Dataset.from_pandas(df_val, preserve_index=False)
ds_test = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
# SAVE CSV-FILES
df_train.to_csv("train.csv",index=False)
df_val.to_csv("val.csv",index=False)
df_test.to_csv("test.csv",index=False)

# LOAD DATASETS TO MAKE SURE THAT THEY ARE AVAILABLE
data_files = {"train": "./modeling/fusion/train_fusion.csv", "val": "./modeling/fusion/val_fusion.csv", "test": "./modeling/fusion/test_fusion.csv"}
dataset = load_dataset("csv", data_files=data_files)

# STORE DATASET FOLDER AND ZIP
path = "./modeling/fusion/ds_fusion"
dataset.save_to_disk(path)
shutil.make_archive(path, 'zip', path)

In [ ]:
import json

id2label = {
    0: "10",
    1: "40",
    2: "50",
    3: "60",
    4: "1140",
    5: "1160",
    6: "1180",
    7: "1280",
    8: "1281",
    9: "1300",
    10: "1301",
    11: "1302",
    12: "1320",
    13: "1560",
    14: "1920",
    15: "1940",
    16: "2060",
    17: "2220",
    18: "2280",
    19: "2403",
    20: "2462",
    21: "2522",
    22: "2582",
    23: "2583",
    24: "2585",
    25: "2705",
    26: "2905"
}

label2id = {
    "10": 0,
    "40": 1,
    "50": 2,
    "60": 3,
    "1140": 4,
    "1160": 5,
    "1180": 6,
    "1280": 7,
    "1281": 8,
    "1300": 9, 
    "1301": 10,
    "1302": 11,
    "1320": 12,
    "1560": 13,
    "1920": 14,
    "1940": 15, 
    "2060": 16,
    "2220": 17,
    "2280": 18,
    "2403": 19,
    "2462": 20,
    "2522": 21,
    "2582": 22,
    "2583": 23,
    "2585": 24,
    "2705": 25,
    "2905": 26
}

with open("id2label", 'w') as json_file:
        json.dump(id2label, json_file, indent=4)

with open("label2id", 'w') as json_file:
        json.dump(label2id, json_file, indent=4)